In [5]:
import os
import numpy as np
import pandas as pd

# ----------------- CONFIGURACIÓN DE RUTAS -----------------
DIR_DATOS = r"C:\Proyectos_de_codigo\Universidad\Sismo\Sismo-geotecnia\datos\datos depurados"

ARCHIVOS = {
    "Sismos_260KM_Depurados_onlymw.csv": "Sismos_260_mw_def.csv",
    "Sismos_50KM_Depurados_onlymw.csv": "Sismos_50_mw_def.csv"
}

def haversine_km(lat1, lon1, lat2, lon2):
    """Calcula distancia usando matemáticas de arreglos rápidos."""
    R = 6371.0
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    delta_phi = np.radians(lat2 - lat1)
    delta_lambda = np.radians(lon2 - lon1)
    a = np.sin(delta_phi / 2.0)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(delta_lambda / 2.0)**2
    return R * (2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a)))

def depurar_duplicados_rapido(df, tol_tiempo_min=60, tol_mag=0.5, tol_dist_km=50.0):
    col_lat = "latitude" if "latitude" in df.columns else "latitud"
    col_lon = "longitude" if "longitude" in df.columns else "longitud"
    col_mw = "escala_mw" if "escala_mw" in df.columns else ("magnitude_mw" if "magnitude_mw" in df.columns else "magnitude")
    
    # Lectura de la hora
    if "hora" in df.columns:
        df["datetime"] = pd.to_datetime(df["hora"], errors="coerce", utc=True)
    elif "fecha_hora" in df.columns:
        df["datetime"] = pd.to_datetime(df["fecha_hora"], errors="coerce", utc=True)
    else:
        df["datetime"] = pd.to_datetime(df["time"], errors="coerce", utc=True)

    # 1. Ordenar por fecha para habilitar la ventana móvil
    df = df.sort_values("datetime").reset_index(drop=True)
    
    # 2. Extraer a Numpy arrays para máxima velocidad en la iteración
    lats = df[col_lat].values
    lons = df[col_lon].values
    mags = df[col_mw].values
    tiempos = df["datetime"].values.astype('datetime64[s]').astype(np.float64) # Segundos
    
    tol_segundos = tol_tiempo_min * 60
    n = len(df)
    to_keep = np.ones(n, dtype=bool) # Array booleano de conservación

    # 3. Iteración optimizada
    for i in range(n):
        if not to_keep[i] or np.isnan(tiempos[i]):
            continue
            
        for j in range(i + 1, n):
            # CORTAR BUCLE: Si el sismo j supera el tiempo límite respecto al i, dejamos de buscar
            if np.isnan(tiempos[j]) or (tiempos[j] - tiempos[i]) > tol_segundos:
                break
                
            if not to_keep[j]:
                continue
                
            # Validación de magnitud y distancia
            if abs(mags[i] - mags[j]) <= tol_mag:
                if haversine_km(lats[i], lons[i], lats[j], lons[j]) <= tol_dist_km:
                    # Conservamos el de mayor magnitud
                    if mags[i] >= mags[j]:
                        to_keep[j] = False
                    else:
                        to_keep[i] = False
                        break # El evento i fue eliminado, pasamos al siguiente i automáticamente
                        
    df_depurado = df[to_keep].copy()
    if "datetime" in df_depurado.columns:
        df_depurado = df_depurado.drop(columns=["datetime"])
        
    return df_depurado, (n - to_keep.sum())

# ==============================================================================
# EJECUCIÓN 
# ==============================================================================

print("Iniciando depuración ultra-rápida...\n")

for entrada, salida in ARCHIVOS.items():
    ruta_in = os.path.join(DIR_DATOS, entrada)
    ruta_out = os.path.join(DIR_DATOS, salida)

    if not os.path.exists(ruta_in):
        print(f"❌ [ERROR] Falta archivo: {ruta_in}")
        continue

    print(f"Procesando: {entrada}...")
    df_orig = pd.read_csv(ruta_in)
    
    # Depurar
    df_limpio, n_eliminados = depurar_duplicados_rapido(df_orig, tol_tiempo_min=60, tol_mag=0.5, tol_dist_km=50.0)
    
    # Guardar
    df_limpio.to_csv(ruta_out, index=False, encoding="utf-8-sig")

    print(f"   - Eventos iniciales  : {len(df_orig)}")
    print(f"   - Duplicados borrados: {n_eliminados}")
    print(f"   - Eventos finales    : {len(df_limpio)}")
    print(f"   ✅ Guardado en       : {salida}\n")

print("Proceso finalizado en segundos.")

Iniciando depuración ultra-rápida...

Procesando: Sismos_260KM_Depurados_onlymw.csv...
   - Eventos iniciales  : 2627
   - Duplicados borrados: 498
   - Eventos finales    : 2129
   ✅ Guardado en       : Sismos_260_mw_def.csv

Procesando: Sismos_50KM_Depurados_onlymw.csv...
   - Eventos iniciales  : 110
   - Duplicados borrados: 16
   - Eventos finales    : 94
   ✅ Guardado en       : Sismos_50_mw_def.csv

Proceso finalizado en segundos.
